# Space-map — CODEX (duodenum) registration

One-click registration of CODEX duodenum serial sections. This file stores
both the registration input (`raw_x`/`raw_y`) and a reference alignment
(`x`/`y`); we register the **raw** coordinates. Runs affine + non-rigid,
then exports an **Imaris multi-layer TIFF** and a **3D reconstruction**.

**Data:** set `DATA` to your local `codex_duodenum.csv.gz`. `LAYERS` limits
how many layers to run (`None` = all).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # this repo's space_map wins

import pandas as pd
from space_map import Slice
from space_map.flow import FlowImport, FlowExport, AutoFlowMultiCenter4

# --- edit this to your local file ---
DATA = 'data/codex_duodenum.csv.gz'
X_COL, Y_COL, LAYER_COL = 'raw_x', 'raw_y', 'layer'
LAYERS = None  # first N layers; None = all
OUT = 'out/codex_duodenum'

## Load per-layer coordinates

CODEX layers are labelled `S1`, `S2`, ... — sort them numerically.

In [ ]:
df = pd.read_csv(DATA, usecols=[X_COL, Y_COL, LAYER_COL])
layer_ids = sorted(df[LAYER_COL].unique(), key=lambda s: int(str(s).lstrip('S')))
if LAYERS:
    layer_ids = layer_ids[:LAYERS]
layers = [df[df[LAYER_COL] == k][[X_COL, Y_COL]].to_numpy(float) for k in layer_ids]
print(f'{len(layers)} layers {layer_ids}, {sum(len(a) for a in layers):,} cells')

## Register (affine + non-rigid)

In [ ]:
fi = FlowImport(f'{OUT}/_work')
slices = fi.init_multi([pd.DataFrame(a, columns=['x', 'y']) for a in layers],
                       ids=[str(k) for k in layer_ids])
af = AutoFlowMultiCenter4(slices, initJKey=Slice.rawKey, alignMethod='auto')
af.affine(useKey='DF', show=False)
af.ldm_pair(Slice.align1Key, Slice.align2Key, show=False)
print('registration done')

## Export: Imaris multi-layer TIFF + 3D reconstruction

In [ ]:
fe = FlowExport(slices)
# Imaris: one single-channel TIFF per layer (1000x1000) -> out/codex_duodenum/imaris/
fe.export_imaris_tiff(f'{OUT}/imaris', key=Slice.align2Key, size=1000)
fe.plot_3d(key=Slice.align2Key, channel='DF',
           save=f'{OUT}/reconstruction_3d.png',
           title='CODEX duodenum - aligned 3D')
print('wrote', f'{OUT}/imaris/', 'and', f'{OUT}/reconstruction_3d.png')

## Inspect the 3D plot inline

In [ ]:
from IPython.display import Image
Image(filename=f'{OUT}/reconstruction_3d.png')

## Command-line equivalent

The same flow can be run without Jupyter (from the `examples/` directory):

```bash
python run_all.py duodenum --data data/codex_duodenum.csv.gz -o out
# or, on your own data, via the space-map CLI:
space-map register data/codex_duodenum.csv.gz --output out/duodenum \
    --x-col raw_x --y-col raw_y --layer-col layer
```

Add `--method sift_vgg` for a lighter, checkpoint-free matcher.